# 04 - Model Training

Trains the primary XGBoost classifier, a Random Forest baseline for comparison, and an Isolation Forest persistence/anomaly layer. Saves all fitted models and encoders to disk.

In [1]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import xgboost as xgb
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

X_train = joblib.load("fe_X_train.joblib")
X_test = joblib.load("fe_X_test.joblib")
y_train = joblib.load("fe_y_train.joblib")
y_test = joblib.load("fe_y_test.joblib")
sample_weights = joblib.load("fe_sample_weights.joblib")
FEATURES = joblib.load("fe_features.joblib")
label_encoder = joblib.load("fe_label_encoder.joblib")
encoders = joblib.load("fe_categorical_encoders.joblib")
df = pd.read_csv("fe_output_df.csv")
print("Loaded train/test splits, encoders, feature list, and dataframe from the feature engineering notebook.")

Loaded train/test splits, encoders, feature list, and dataframe from the feature engineering notebook.


## Step 5: Train XGBoost (primary model)

In [2]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softprob", num_class=len(label_encoder.classes_),
    eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

xgb_train_pred = xgb_model.predict(X_train)
xgb_pred = xgb_model.predict(X_test)

print(classification_report(y_test, xgb_pred, target_names=label_encoder.classes_))

                            precision    recall  f1-score   support

              agricultural       1.00      1.00      1.00      4770
                     flare       0.94      0.99      0.96       713
                industrial       0.95      0.96      0.96      2303
                    mining       0.98      1.00      0.99      3034
offshore_flare_or_platform       0.57      0.55      0.56        93
                  wildfire       0.99      0.97      0.98      6783

                  accuracy                           0.98     17696
                 macro avg       0.90      0.91      0.91     17696
              weighted avg       0.98      0.98      0.98     17696



## Step 6: Train Random Forest (baseline comparison)

In [3]:
rf_model = RandomForestClassifier(
    n_estimators=400, max_depth=14, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred, target_names=label_encoder.classes_))

                            precision    recall  f1-score   support

              agricultural       1.00      0.99      0.99      4770
                     flare       0.92      0.99      0.96       713
                industrial       0.92      0.99      0.95      2303
                    mining       0.97      1.00      0.99      3034
offshore_flare_or_platform       0.74      0.27      0.39        93
                  wildfire       1.00      0.97      0.98      6783

                  accuracy                           0.98     17696
                 macro avg       0.92      0.87      0.88     17696
              weighted avg       0.98      0.98      0.98     17696



## Step 8: Persistence / anomaly layer

Answers a DIFFERENT question than the classifier: not "what type of fire is this",
but "is this location's current behaviour abnormal compared to its own history" —
e.g. a stable flare suddenly running much hotter, or a stable vegetation signature
(NDVI/NBR) suddenly dropping.

In [4]:
anomaly_features = ["bright_ti4", "frp", "persistence_count_30d", "frp_to_temp_ratio", "ndvi", "nbr"]
iso_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=RANDOM_STATE)
df["anomaly_score"] = iso_model.fit_predict(df[anomaly_features])
df["is_anomalous"] = df["anomaly_score"] == -1

print(f"Flagged {df['is_anomalous'].sum()} rows ({df['is_anomalous'].mean()*100:.1f}%) as anomalous")
df.groupby("category")["is_anomalous"].mean().sort_values(ascending=False)

Flagged 4644 rows (5.0%) as anomalous


category
wildfire                      0.089716
mining                        0.036848
agricultural                  0.026097
flare                         0.019355
offshore_flare_or_platform    0.019108
industrial                    0.015378
Name: is_anomalous, dtype: float64

## Step 9: Save everything

In [5]:
joblib.dump(xgb_model, "model_xgboost_v2.joblib")
joblib.dump(rf_model, "model_random_forest_v2.joblib")
joblib.dump(iso_model, "model_isolation_forest_v2.joblib")
joblib.dump(label_encoder, "label_encoder_v2.joblib")
joblib.dump(encoders, "categorical_encoders_v2.joblib")
joblib.dump(FEATURES, "feature_list_v2.joblib")

df.to_csv("firms_combined_with_predictions_v2.csv", index=False)
print("Saved all v2 models + firms_combined_with_predictions_v2.csv")

Saved all v2 models + firms_combined_with_predictions_v2.csv


### Save additional artifacts for the evaluation notebook

In [6]:
# --- Notebook-chaining glue code (added when splitting the original pipeline) ---
joblib.dump(xgb_train_pred, "train_xgb_train_pred.joblib")
joblib.dump(xgb_pred, "train_xgb_pred.joblib")
joblib.dump(rf_train_pred, "train_rf_train_pred.joblib")
joblib.dump(rf_pred, "train_rf_pred.joblib")
joblib.dump(y_train, "train_y_train.joblib")
joblib.dump(y_test, "train_y_test.joblib")
joblib.dump(FEATURES, "train_features.joblib")
joblib.dump(label_encoder, "train_label_encoder.joblib")
print("Saved prediction arrays and metadata for the evaluation notebook.")

Saved prediction arrays and metadata for the evaluation notebook.
